In [2]:
# Cell 1: Install dependencies
!pip install -q gradio scikit-learn pandas numpy matplotlib seaborn
print("✅ Libraries installed successfully!")

✅ Libraries installed successfully!


In [3]:
# Cell 2: Data Loading & Preprocessing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import gradio as gr

print("⚙️ Tabular Dataset Load ho raha hai...")
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target
target_names = iris.target_names

# Plotting ke liye target names ko dataframe mein add kar rahe hain
X['species'] = [target_names[i] for i in y]

# Train-Test Split (80/20)
print("✂️ Data ko 80% Training aur 20% Testing mein split kar rahe hain...")
X_features = X.drop('species', axis=1)
X_train, X_test, y_train, y_test = train_test_split(X_features, y, test_size=0.2, random_state=42)

# Scaling (Data ko balance karna)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("✅ Data Preprocessing Complete!")

⚙️ Tabular Dataset Load ho raha hai...
✂️ Data ko 80% Training aur 20% Testing mein split kar rahe hain...
✅ Data Preprocessing Complete!


In [4]:
# Cell 3: Model Training & Evaluation
print("🧠 Supervised Learning Model (KNN) Train ho raha hai...")
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train_scaled, y_train)

print("\n📊 Model Performance Check:")
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"🎯 Model Accuracy: {accuracy * 100:.2f}%")

print("\nConfusion Matrix (TP, FP, FN, TN check karne ke liye):")
print(confusion_matrix(y_test, y_pred))

🧠 Supervised Learning Model (KNN) Train ho raha hai...

📊 Model Performance Check:
🎯 Model Accuracy: 100.00%

Confusion Matrix (TP, FP, FN, TN check karne ke liye):
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]


In [ ]:
import gradio as gr
# Cell 4: Frontend UI with Real-Time Plots
def predict_and_plot(sepal_length, sepal_width, petal_length, petal_width):
    # 1. Perform Prediction
    input_data = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
    scaled_input = scaler.transform(input_data)
    prediction = model.predict(scaled_input)
    predicted_class = target_names[prediction[0]].upper()

    # 2. Real-time Plotting (Matplotlib & Seaborn)
    # Upgraded to a 2x2 grid to accommodate 4 plots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Plot 1: Sepal Length vs Sepal Width
    sns.scatterplot(data=X, x='sepal length (cm)', y='sepal width (cm)', hue='species', ax=axes[0, 0], palette='viridis', alpha=0.5)
    # Highlight user input with a Red Star
    axes[0, 0].scatter(sepal_length, sepal_width, color='red', s=300, marker='*', edgecolor='black', label='Your Input')
    axes[0, 0].set_title("Sepal Features (Length vs Width)")
    axes[0, 0].legend()

    # Plot 2: Petal Length vs Petal Width
    sns.scatterplot(data=X, x='petal length (cm)', y='petal width (cm)', hue='species', ax=axes[0, 1], palette='viridis', alpha=0.5)
    # Highlight user input with a Red Star
    axes[0, 1].scatter(petal_length, petal_width, color='red', s=300, marker='*', edgecolor='black', label='Your Input')
    axes[0, 1].set_title("Petal Features (Length vs Width)")
    axes[0, 1].legend()

    # Plot 3: Sepal Length vs Petal Length
    sns.scatterplot(data=X, x='sepal length (cm)', y='petal length (cm)', hue='species', ax=axes[1, 0], palette='viridis', alpha=0.5)
    # Highlight user input with a Red Star
    axes[1, 0].scatter(sepal_length, petal_length, color='red', s=300, marker='*', edgecolor='black', label='Your Input')
    axes[1, 0].set_title("Sepal Length vs Petal Length")
    axes[1, 0].legend()

    # Plot 4: Sepal Width vs Petal Width
    sns.scatterplot(data=X, x='sepal width (cm)', y='petal width (cm)', hue='species', ax=axes[1, 1], palette='viridis', alpha=0.5)
    # Highlight user input with a Red Star
    axes[1, 1].scatter(sepal_width, petal_width, color='red', s=300, marker='*', edgecolor='black', label='Your Input')
    axes[1, 1].set_title("Sepal Width vs Petal Width")
    axes[1, 1].legend()

    plt.tight_layout()

    return f"AI Prediction: {predicted_class}", fig

# Advanced UI Design (Blocks API)
with gr.Blocks(theme=gr.themes.Soft()) as ui:
    gr.Markdown("<h1 style='text-align: center;'>🌸 DecodeLabs | Project 2 Classification Engine</h1>")
    gr.Markdown("<p style='text-align: center;'>Adjust your values using the sliders and see where your data point fits in the real-time graphs (Red Star ⭐).</p>")

    with gr.Row():
        # Left Side: Inputs (Sliders)
        with gr.Column(scale=1):
            sl = gr.Slider(0.0, 10.0, value=5.1, step=0.1, label="Sepal Length (cm)")
            sw = gr.Slider(0.0, 10.0, value=3.5, step=0.1, label="Sepal Width (cm)")
            pl = gr.Slider(0.0, 10.0, value=1.4, step=0.1, label="Petal Length (cm)")
            pw = gr.Slider(0.0, 10.0, value=0.2, step=0.1, label="Petal Width (cm)")
            btn = gr.Button("Predict & Generate Plots 🚀", variant="primary")

        # Right Side: Outputs (Text & Graph)
        with gr.Column(scale=2):
            output_text = gr.Textbox(label="AI Output Layer", lines=1)
            output_plot = gr.Plot(label="Real-Time Data Visualization (4 Metrics)")

    # Connect button click event
    btn.click(fn=predict_and_plot, inputs=[sl, sw, pl, pw], outputs=[output_text, output_plot])

# Launch the UI
print("\n🌐 Launching Graphical User Interface...")
ui.launch(share=True, debug=True)

/tmp/ipykernel_431/3111984049.py:47: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as ui:



🌐 Launching Graphical User Interface...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f1079c132c7156f986.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
